# 01 — Análisis Exploratorio de Datos (EDA)

Reproduce el informe v1.1 sobre el corpus salvadoreño.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from discurso_odio.config import load_settings
from discurso_odio.data.loaders import load_all_splits, load_split, get_class_weights
from discurso_odio.paths import REPORTS_FIGURES, REPORTS_METRICS
from discurso_odio.utils.plotting import setup_plot_style, get_label_names, get_class_palette
from discurso_odio.utils.seed import set_seed

settings = load_settings()
set_seed(settings['project']['seed'])
setup_plot_style()
LABELS = get_label_names()
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS.mkdir(parents=True, exist_ok=True)
print('Proyecto:', settings['project']['name'])



In [ ]:
splits = load_all_splits()
corpus = splits['corpus']
train, val, test = splits['train'], splits['val'], splits['test']
print(f'Corpus: {len(corpus)} | Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')
corpus.head(3)


## 2. Auditoría de integridad


In [ ]:
audit = {
    'filas_totales': len(corpus),
    'ids_duplicados': corpus['ID'].duplicated().sum(),
    'textos_duplicados': corpus['texto_modelo'].duplicated().sum(),
    'usuarios_repetidos': corpus['Usuario_ID_Anonimizado'].duplicated().sum(),
}
pd.Series(audit).to_frame('valor')


## 3. Distribución de clases


In [ ]:
class_dist = corpus['clase'].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
class_dist.plot(kind='bar', color=get_class_palette(), ax=ax)
ax.set_title('Distribución de clases canónicas')
ax.set_ylabel('Registros')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / '01_distribucion_clases.png')
plt.show()
class_dist


## 3.1 Pisos de referencia sin modelo


In [ ]:
from discurso_odio.evaluation.metrics import trivial_baselines
counts = corpus['label'].value_counts().to_dict()
baselines = trivial_baselines(counts)
pd.Series(baselines, name='F1-macro').to_frame()


## 4. Análisis cross-platform


In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu

ct = pd.crosstab(corpus['Plataforma'], corpus['clase'])
chi2, p, dof, _ = chi2_contingency(ct)
n = ct.sum().sum()
cramers_v = (chi2 / (n * (min(ct.shape) - 1))) ** 0.5

x_len = corpus[corpus['Plataforma']=='X']['n_palabras']
fb_len = corpus[corpus['Plataforma']=='Facebook']['n_palabras']
u_stat, u_p = mannwhitneyu(x_len, fb_len, alternative='two-sided')

pd.DataFrame({
    'prueba': ['Chi-cuadrado', 'V de Cramér', 'Mann-Whitney U (longitud)'],
    'estadistico': [f'{chi2:.1f}', f'{cramers_v:.3f}', f'{u_stat:.0f}'],
    'p_valor': [f'{p:.2e}', '-', f'{u_p:.2e}'],
})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pd.crosstab(corpus['Plataforma'], corpus['clase'], normalize='index').plot(
    kind='bar', stacked=True, ax=axes[0], color=get_class_palette())
axes[0].set_title('Composición por plataforma')
corpus.boxplot(column='n_palabras', by='Plataforma', ax=axes[1])
axes[1].set_title('Longitud por plataforma')
plt.suptitle('')
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / '02_plataforma_clases.png')
plt.show()


## 5. Caracterización lingüística


In [ ]:
dialect_stats = {
    'textos_con_jerga': f"{(corpus['n_jerga']>0).mean()*100:.1f}%",
    'media_jerga': corpus['n_jerga'].mean(),
    'lift_toxicidad_jerga': (
        (corpus[corpus['n_jerga']>0]['label']>0).mean() /
        max((corpus[corpus['n_jerga']==0]['label']>0).mean(), 1e-9)
    ),
}
pd.Series(dialect_stats, name='valor')


## 7. Tokenización mBERT — cobertura max_length


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(settings['model']['name'])
lengths = [len(tokenizer.encode(t, add_special_tokens=True)) for t in corpus['texto_modelo'].head(500)]
p99 = np.percentile(lengths, 99)
print(f'Percentil 99 tokens: {p99:.0f} | max_length configurado: {settings["model"]["max_length"]}')
plt.hist(lengths, bins=30, color='#3498db', edgecolor='white')
plt.axvline(settings['model']['max_length'], color='red', linestyle='--', label='max_length=128')
plt.title('Distribución de longitud en tokens (muestra 500)')
plt.xlabel('Tokens')
plt.legend()
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / '03_token_lengths.png')
plt.show()
